# VORTEX-AI (CG-NSDE) - Colab GPU run (one click)

**Runtime -> Run all.** Nothing else to do.

What runs automatically:

1. checks the GPU
2. installs only the missing packages - Colab's CUDA-matched PyTorch is never replaced
3. downloads the private `vortex-ai-code` and `vortex-ai-data` Kaggle datasets
4. links the 374 MB artifact set in as `data/raw`, so nothing is regenerated
5. trains, copying checkpoints to Drive every 2 minutes so a disconnect does not lose the run
6. evaluates the best checkpoint
7. copies checkpoints, logs and metrics to Drive

---

### One-time setup (30 seconds, first run only)

The datasets are private, so the notebook needs your Kaggle token.

Left sidebar -> key icon (**Secrets**) -> *Add a new secret*:

| Field | Value |
| --- | --- |
| Name | `KAGGLE_API_TOKEN` |
| Value | your `KGAT_...` token |

Tick **Notebook access**, then Runtime -> Run all. Every later run is genuinely zero-click.

Set `SMOKE = True` in the next cell for a 2-epoch validation run instead of the full 200.

> Runtime is ~3.5 h on a T4. Free Colab can reclaim a session; the Drive sync is what makes
> that survivable. Colab Pro's background execution is the safer home for a run this long.

## 1. Configuration

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
import threading
import time

SMOKE = False          # True -> 2-epoch validation run
MAX_EPOCHS = 2 if SMOKE else 200
BATCH_SIZE = 4
NUM_WORKERS = 2
USE_DRIVE = True       # checkpoint sync + final copy; survives a disconnect

CODE_DATASET = "brxkeaaryan/vortex-ai-code"
DATA_DATASET = "brxkeaaryan/vortex-ai-data"
WORK = "/content/vortex-ai"
DRIVE_OUT = "/content/drive/MyDrive/vortex_ai_runs"
RUN_TAG = "colab_smoke" if SMOKE else "colab_t4"

print(f"max_epochs={MAX_EPOCHS} batch_size={BATCH_SIZE} workers={NUM_WORKERS} smoke={SMOKE}")

## 2. GPU check

In [ ]:
import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus", torch.cuda.device_count())
if torch.cuda.is_available():
    print(" ", torch.cuda.get_device_name(0))
    print("  vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

## 3. Dependencies

Only missing packages are installed. `torch-geometric` comes from plain PyPI because the repo uses
`GATConv` and `dense_to_sparse` only, not the `torch_scatter` / `torch_sparse` compiled extensions.

In [ ]:
REQUIRED = {
    "torch_geometric": "torch-geometric",
    "torchsde": "torchsde",
    "pytorch_lightning": "pytorch-lightning",
    "wandb": "wandb",
    "kaggle": "kaggle",
}

missing = []
for module, package in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)

if missing:
    print("installing:", missing, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies already present")

import pytorch_lightning
import torch_geometric
import torchsde

print("torch_geometric", torch_geometric.__version__)
print("torchsde", torchsde.__version__)
print("pytorch_lightning", pytorch_lightning.__version__)

## 4. Fetch code and data

Both come from the Kaggle datasets so Colab gets exactly the arrays used locally - no
re-downloading prices, and none of the ticker-rename breakage that makes regeneration unsafe.

In [ ]:
def kaggle_token():
    token = os.environ.get("KAGGLE_API_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        return None


token = kaggle_token()
if not token:
    raise SystemExit(
        "Missing KAGGLE_API_TOKEN.\n"
        "  Left sidebar -> key icon -> Add a new secret\n"
        "    Name : KAGGLE_API_TOKEN\n"
        "    Value: your KGAT_... token\n"
        "  tick 'Notebook access', then Runtime -> Run all again."
    )
os.environ["KAGGLE_API_TOKEN"] = token

DL = "/content/dl"
shutil.rmtree(DL, ignore_errors=True)
os.makedirs(DL, exist_ok=True)

for ref in (CODE_DATASET, DATA_DATASET):
    print(f"\n$ kaggle datasets download -d {ref}", flush=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", ref, "-p", DL, "--unzip", "--quiet"],
        check=True,
    )

print("\ndownloaded:", sorted(os.listdir(DL)))

## 5. Stage the workspace

Kaggle unpacks archives server-side, so the code arrives as a `repo/` tree. The data mount is
symlinked in rather than copied, which avoids 374 MB of disk churn per run.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)

src = os.path.join(DL, "repo")
if not os.path.isdir(src):
    hits = glob.glob(os.path.join(DL, "**", "train_vortex.py"), recursive=True)
    assert hits, "train_vortex.py not found in the code download"
    src = os.path.dirname(hits[0])
shutil.copytree(src, WORK)
os.chdir(WORK)

data_hits = glob.glob(os.path.join(DL, "**", "windows.npy"), recursive=True)
assert data_hits, "windows.npy not found in the data download"
data_dir = os.path.dirname(data_hits[0])

raw = os.path.join(WORK, "data", "raw")
if os.path.islink(raw):
    os.unlink(raw)
elif os.path.isdir(raw):
    shutil.rmtree(raw)
os.symlink(data_dir, raw)

print("workdir:", os.getcwd())
print("data/raw ->", os.path.realpath(raw))

required = [
    "train_vortex.py",
    "evaluate_vortex.py",
    "training/config.yaml",
    "models/gat_encoder.py",
    "models/neural_sde.py",
    "data/raw/nifty50_close.csv",
]
absent = [f for f in required if not os.path.exists(f)]
assert not absent, f"missing: {absent}"
print("all required files present")

In [ ]:
import numpy as np

windows = np.load("data/raw/windows.npy")
adj = np.load("data/raw/adj_matrices.npy")
feats = np.load("data/raw/window_node_features.npy")

print("windows     ", windows.shape)
print("adj_matrices", adj.shape)
print("node_feats  ", feats.shape)

n, T, N = windows.shape
assert (T, N) == (60, 50), f"unexpected window shape {windows.shape}"
assert adj.shape == (n, N, N), f"unexpected adjacency shape {adj.shape}"
# node features come from build_scaler_split and can be one window shorter; train_vortex.py
# aligns all three with min() at load time.
assert feats.shape[1:] == (T, N, 6), f"unexpected node-feature shape {feats.shape}"
assert feats.shape[0] in (n, n - 1), f"node-feature length {feats.shape[0]} vs {n} windows"

for name in ("scaler.npz", "split.npz", "window_regimes_v2.npy", "volume_windows.npy"):
    assert os.path.exists(f"data/raw/{name}"), f"missing artifact: {name}"

print(f"OK: {n} windows x {T} days x {N} stocks")

## 6. Train

A background thread copies new checkpoints to Drive every 2 minutes, so a Colab disconnect costs
at most two epochs of progress. `--devices 1` on purpose: the model is tiny (50 nodes, batch 4).

In [ ]:
drive_ckpt = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_ckpt = os.path.join(DRIVE_OUT, "checkpoints")
        os.makedirs(drive_ckpt, exist_ok=True)
        print("drive checkpoints:", drive_ckpt)
    except Exception as exc:
        print("drive unavailable; checkpoints will not survive a disconnect:", exc)

stop = threading.Event()


def sync_checkpoints():
    while not stop.is_set():
        if drive_ckpt:
            for path in glob.glob(os.path.join(WORK, "models/checkpoints/*.ckpt")):
                try:
                    shutil.copy2(path, drive_ckpt)
                except Exception:
                    pass
        stop.wait(120)


thread = threading.Thread(target=sync_checkpoints, daemon=True)
thread.start()

cmd = [
    "python", "train_vortex.py",
    "--config", "training/config.yaml",
    "--accelerator", "cuda",
    "--devices", "1",
    "--max_epochs", str(MAX_EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--run_tag", RUN_TAG,
]

print("$ " + " ".join(cmd), flush=True)
try:
    subprocess.run(cmd, check=True, cwd=WORK)
finally:
    stop.set()
    thread.join(timeout=15)
    print("checkpoint sync stopped")

## 7. Evaluate the best checkpoint

Skipped in smoke mode.

In [ ]:
if SMOKE:
    print("smoke run: skipping evaluation")
else:
    checkpoints = glob.glob(os.path.join(WORK, "models/checkpoints/*.ckpt"))
    assert checkpoints, "no checkpoints were written"
    best = max(checkpoints, key=os.path.getmtime)
    print("checkpoint:", best)
    os.makedirs(os.path.join(WORK, "results"), exist_ok=True)
    subprocess.run(
        [
            "python", "evaluate_vortex.py",
            "--checkpoint", best,
            "--device", "cuda",
            "--n_scenarios", "500",
            "--batch_size", "4",
            "--output", "results/eval_colab.json",
        ],
        check=True,
        cwd=WORK,
    )

## 8. Save outputs

In [ ]:
saved = False
if USE_DRIVE and os.path.isdir("/content/drive/MyDrive"):
    for sub in ("models/checkpoints", "runs", "results"):
        source = os.path.join(WORK, sub)
        if os.path.isdir(source):
            target = os.path.join(DRIVE_OUT, sub)
            shutil.rmtree(target, ignore_errors=True)
            shutil.copytree(source, target)
            print("copied", sub, "->", target)
    saved = True
else:
    print("drive not mounted; outputs stay under", WORK)

print()
for root, _, files in os.walk(WORK):
    for f in files:
        if f.endswith((".ckpt", ".json", ".csv")):
            path = os.path.join(root, f)
            print(f"{os.path.getsize(path) / 1e6:9.2f} MB  {os.path.relpath(path, WORK)}")

print("\nsaved to drive:", saved)